# 14 — Extended Baseline Comparison

**Goal:** Compare MARS against 18 baselines across three evaluation dimensions.

**Three tables for the paper:**

| Table | Dimension | Baselines |
|-------|-----------|-----------|
| 1 | Knowledge Gap Prediction | BKT, DKT, SAKT, SAINT, AKT/DKT+, Moving Average, MARS |
| 2 | Content Recommendation | Random, Popular, CF-only, Content-only, BPR-MF, NeuMF, KG-RS, MARS |
| 3 | Strategy Selection (Bandits) | Thompson Sampling, e-greedy, UCB1, Fixed |

**Protocol:** 5 seeds, Wilcoxon signed-rank + Bonferroni correction, same chronological test split.

In [ ]:
"""Cell 1 — Imports and setup."""

from __future__ import annotations

import json
import logging
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats as sp_stats
from scipy.stats import wilcoxon
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score, roc_auc_score,
)

warnings.filterwarnings("ignore")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from agents.utils import set_global_seed, load_config

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("baselines")
logger.setLevel(logging.INFO)

SEEDS = [42, 123, 456, 789, 2024]
NUM_TAGS = 293
SEQ_LEN = 50
HORIZON = 10
TOP_K = 10
RESULTS_DIR = ROOT / "results" / "baselines"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = load_config(str(ROOT / "configs" / "config.yaml"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Seeds: {SEEDS}")

In [ ]:
"""Cell 2 — Data loading and eval_pairs construction."""

from data.loader import EdNetLoader
from data.preprocessor import EdNetPreprocessor

splits_dir = ROOT / "data" / "splits"

if (splits_dir / "train.parquet").exists():
    train_df = pd.read_parquet(splits_dir / "train.parquet")
    val_df = pd.read_parquet(splits_dir / "val.parquet")
    test_df = pd.read_parquet(splits_dir / "test.parquet")
else:
    loader = EdNetLoader(data_dir=str(ROOT / "data" / "raw"))
    interactions = loader.load_interactions(
        sample_users=CONFIG.get("data", {}).get("sample_users", 50000),
    )
    preprocessor = EdNetPreprocessor()
    splits = preprocessor.run(interactions, chunked=True)
    train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

loader = EdNetLoader(data_dir=str(ROOT / "data" / "raw"))
questions_df = loader.questions
lectures_df = loader.lectures

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Test users: {test_df['user_id'].nunique()}")


def parse_tags(tags):
    """Parse tags field into list of ints."""
    if isinstance(tags, list):
        return [int(t) for t in tags if 0 <= int(t) < NUM_TAGS]
    if isinstance(tags, (int, np.integer)):
        return [int(tags)] if 0 <= int(tags) < NUM_TAGS else []
    if isinstance(tags, str):
        return [int(t) for t in tags.split(";") if t.strip().isdigit() and 0 <= int(t) < NUM_TAGS]
    return []


def build_eval_pairs(test_df, context_ratio=0.5):
    """Build evaluation pairs from test data."""
    eval_pairs = []
    for uid, grp in test_df.groupby("user_id"):
        grp = grp.sort_values("timestamp")
        split_idx = max(1, int(len(grp) * context_ratio))
        ctx = grp.iloc[:split_idx]
        future = grp.iloc[split_idx:]
        if len(future) == 0:
            continue

        # Ground truth: tags the user failed on
        gt = set()
        gt_all = set()
        for _, row in future.iterrows():
            tags = parse_tags(row["tags"])
            gt_all.update(tags)
            if not row["correct"]:
                gt.update(tags)

        if len(gt) == 0:
            continue
        eval_pairs.append((str(uid), ctx, gt, gt_all, 0))

    return eval_pairs


eval_pairs = build_eval_pairs(test_df)
print(f"Eval pairs: {len(eval_pairs)}")

In [ ]:
"""Cell 3 — Metrics computation (shared across all baselines)."""


def compute_all_metrics(preds, scores_list, gt_list, gta_list, top_k=10):
    """
    Compute all metrics for a baseline.

    Parameters
    ----------
    preds : list[list[int]]
        Ranked tag IDs per user (descending by score).
    scores_list : list[np.ndarray]
        Score arrays of shape (NUM_TAGS,) per user.
    gt_list : list[set[int]]
        Ground truth failed tags per user.
    gta_list : list[set[int]]
        All ground truth tags per user.

    Returns
    -------
    dict with keys: auc_roc, auc_pr, f1_macro, ndcg@5, ndcg@10, precision@10,
                    recall@10, map, mrr, coverage
    """
    metrics = {}

    # --- AUC-ROC, AUC-PR, F1-macro (tag-level binary classification) ---
    y_true_all = []
    y_score_all = []
    for scores, gt in zip(scores_list, gt_list):
        binary = np.zeros(NUM_TAGS)
        for t in gt:
            if 0 <= t < NUM_TAGS:
                binary[t] = 1.0
        y_true_all.append(binary)
        y_score_all.append(scores)

    y_true = np.array(y_true_all)
    y_score = np.array(y_score_all)

    # Mask: only tags with at least one positive across users
    col_mask = y_true.sum(axis=0) > 0
    if col_mask.any():
        try:
            metrics["auc_roc"] = float(roc_auc_score(
                y_true[:, col_mask], y_score[:, col_mask], average="macro"
            ))
        except ValueError:
            metrics["auc_roc"] = 0.0
        try:
            metrics["auc_pr"] = float(average_precision_score(
                y_true[:, col_mask], y_score[:, col_mask], average="macro"
            ))
        except ValueError:
            metrics["auc_pr"] = 0.0
    else:
        metrics["auc_roc"] = 0.0
        metrics["auc_pr"] = 0.0

    # F1-macro (threshold = 0.5)
    y_pred_bin = (y_score >= 0.5).astype(int)
    row_mask = y_true.sum(axis=1) > 0
    if row_mask.any():
        metrics["f1_macro"] = float(f1_score(
            y_true[row_mask].ravel(), y_pred_bin[row_mask].ravel(),
            average="macro", zero_division=0,
        ))
    else:
        metrics["f1_macro"] = 0.0

    # --- Ranking metrics ---
    ndcg5s, ndcg10s, p10s, r10s, aps, rrs = [], [], [], [], [], []
    all_recommended = set()

    for ranked, gt in zip(preds, gt_list):
        ranked_k = ranked[:top_k]
        ranked_5 = ranked[:5]
        all_recommended.update(ranked_k)

        hits_10 = len(set(ranked_k) & gt)
        hits_5 = len(set(ranked_5) & gt)

        # Precision@K, Recall@K
        p10s.append(hits_10 / top_k)
        r10s.append(hits_10 / len(gt) if gt else 0.0)

        # NDCG@5
        dcg5 = sum(1.0 / np.log2(r + 2) for r, t in enumerate(ranked_5) if t in gt)
        ideal5 = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), 5)))
        ndcg5s.append(dcg5 / ideal5 if ideal5 > 0 else 0.0)

        # NDCG@10
        dcg10 = sum(1.0 / np.log2(r + 2) for r, t in enumerate(ranked_k) if t in gt)
        ideal10 = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), top_k)))
        ndcg10s.append(dcg10 / ideal10 if ideal10 > 0 else 0.0)

        # MAP (Average Precision)
        ap_hits = 0
        ap_sum = 0.0
        for r, t in enumerate(ranked_k):
            if t in gt:
                ap_hits += 1
                ap_sum += ap_hits / (r + 1)
        aps.append(ap_sum / min(len(gt), top_k) if gt else 0.0)

        # MRR (Mean Reciprocal Rank)
        rr = 0.0
        for r, t in enumerate(ranked_k):
            if t in gt:
                rr = 1.0 / (r + 1)
                break
        rrs.append(rr)

    metrics["ndcg@5"] = float(np.mean(ndcg5s))
    metrics["ndcg@10"] = float(np.mean(ndcg10s))
    metrics["precision@10"] = float(np.mean(p10s))
    metrics["recall@10"] = float(np.mean(r10s))
    metrics["map"] = float(np.mean(aps))
    metrics["mrr"] = float(np.mean(rrs))
    metrics["coverage"] = len(all_recommended) / NUM_TAGS

    return {k: round(v, 4) for k, v in metrics.items()}


print("Metrics function ready: auc_roc, auc_pr, f1_macro, ndcg@5/10, p@10, r@10, map, mrr, coverage")

In [ ]:
"""Cell 4 — Existing baselines (from 08_evaluation): Random, Popular, CF-only, Content-only."""


def run_baseline(baseline_fn, eval_pairs, **kwargs):
    """Run a baseline and compute all metrics."""
    preds, scores_list, gt_list, gta_list = baseline_fn(eval_pairs, **kwargs)
    return compute_all_metrics(preds, scores_list, gt_list, gta_list)


# --- Random ---
def baseline_random(eval_pairs, seed=42):
    rng = np.random.RandomState(seed)
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        scores = rng.random(NUM_TAGS).astype(np.float32)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# --- Popular (global tag failure frequency) ---
def baseline_popular(eval_pairs, train_df=None):
    tag_fail = np.zeros(NUM_TAGS, dtype=np.float32)
    tag_total = np.zeros(NUM_TAGS, dtype=np.float32)
    for _, row in train_df.iterrows():
        for t in parse_tags(row["tags"]):
            tag_total[t] += 1
            if not row["correct"]:
                tag_fail[t] += 1
    scores_global = np.divide(tag_fail, tag_total, out=np.zeros_like(tag_fail),
                              where=tag_total > 0)
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        ranked = np.argsort(scores_global)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores_global.copy())
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# --- Collaborative Filtering (SVD) ---
def baseline_cf_only(eval_pairs, train_df=None, seed=42):
    from sklearn.decomposition import TruncatedSVD
    from scipy.sparse import lil_matrix

    user_ids = train_df["user_id"].unique()
    uid_map = {u: i for i, u in enumerate(user_ids)}
    mat = lil_matrix((len(user_ids), NUM_TAGS), dtype=np.float32)

    for uid, grp in train_df.groupby("user_id"):
        idx = uid_map[uid]
        for _, row in grp.iterrows():
            for t in parse_tags(row["tags"]):
                if not row["correct"]:
                    mat[idx, t] += 1.0

    n_comp = min(32, min(mat.shape) - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed)
    user_factors = svd.fit_transform(mat.tocsr())
    item_factors = svd.components_.T

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        if uid in uid_map:
            u_idx = uid_map[uid]
            scores = (item_factors @ user_factors[u_idx]).astype(np.float32)
        else:
            scores = np.zeros(NUM_TAGS, dtype=np.float32)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# --- Content-only (tag difficulty + user failure rate blend) ---
def baseline_content_only(eval_pairs, train_df=None, seed=42):
    tag_fail = np.zeros(NUM_TAGS, dtype=np.float32)
    tag_total = np.zeros(NUM_TAGS, dtype=np.float32)
    for _, row in train_df.iterrows():
        for t in parse_tags(row["tags"]):
            tag_total[t] += 1
            if not row["correct"]:
                tag_fail[t] += 1
    tag_difficulty = np.divide(tag_fail, tag_total, out=np.full(NUM_TAGS, 0.5),
                               where=tag_total > 0)

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        user_fail = np.zeros(NUM_TAGS, dtype=np.float32)
        user_total = np.zeros(NUM_TAGS, dtype=np.float32)
        for _, row in ctx.iterrows():
            for t in parse_tags(row["tags"]):
                user_total[t] += 1
                if not row["correct"]:
                    user_fail[t] += 1
        user_rate = np.divide(user_fail, user_total, out=np.full(NUM_TAGS, 0.5),
                              where=user_total > 0)
        scores = (0.5 * tag_difficulty + 0.5 * user_rate).astype(np.float32)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# --- IRT-only ---
def baseline_irt_only(eval_pairs, train_df=None, seed=42):
    from agents.diagnostic_agent import DiagnosticAgent
    diag = DiagnosticAgent(seed=seed)
    diag.calibrate_from_interactions(train_df, min_answers_per_q=20)

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        result = diag.run_assessment(uid, ctx)
        theta = result.get("theta", 0.0)
        # Score = P(fail) per tag using IRT 3PL
        scores = np.zeros(NUM_TAGS, dtype=np.float32)
        if hasattr(diag, "_irt_params") and diag._irt_params is not None:
            for i, qid in enumerate(diag._irt_params):
                tag_map = getattr(diag, "_tag_by_item", {})
                tags_for_q = tag_map.get(i, [])
                a = diag._a[i] if hasattr(diag, "_a") else 1.0
                b = diag._b[i] if hasattr(diag, "_b") else 0.0
                c = diag._c[i] if hasattr(diag, "_c") else 0.25
                p_correct = c + (1 - c) / (1 + np.exp(-a * (theta - b)))
                for t in tags_for_q:
                    if 0 <= t < NUM_TAGS:
                        scores[t] = max(scores[t], 1.0 - p_correct)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("Existing baselines ready: Random, Popular, CF-only, Content-only, IRT-only")

## Knowledge Tracing Baselines

DKT, SAKT (existing) + BKT, SAINT, DKT+, Moving Average (new).

In [ ]:
"""Cell 5 — DKT and SAKT models (from 08_evaluation)."""


class DKTModel(nn.Module):
    """Deep Knowledge Tracing — LSTM baseline."""

    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags + 1, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim + 1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, n_tags)

    def forward(self, tag_ids, correct):
        emb = self.tag_emb(tag_ids)                        # (B, T, 32)
        x = torch.cat([emb, correct.unsqueeze(-1)], dim=-1)  # (B, T, 33)
        out, _ = self.lstm(x)                               # (B, T, 64)
        logits = self.fc(out[:, -1, :])                     # (B, 293)
        return torch.sigmoid(logits)


class SAKTModel(nn.Module):
    """Self-Attentive Knowledge Tracing — Transformer baseline."""

    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, d_model=64, nhead=4,
                 num_layers=2, max_len=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags + 1, emb_dim, padding_idx=0)
        self.input_proj = nn.Linear(emb_dim + 1, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model,
            dropout=0.1, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, n_tags)

    def forward(self, tag_ids, correct):
        emb = self.tag_emb(tag_ids)
        x = torch.cat([emb, correct.unsqueeze(-1)], dim=-1)
        x = self.input_proj(x)
        T = x.size(1)
        pos = torch.arange(T, device=x.device).clamp(0, 63).unsqueeze(0).expand(x.size(0), -1)
        x = x + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(x, mask=mask)
        return torch.sigmoid(self.fc(out[:, -1, :]))


def build_kt_sequences(df, seq_len=SEQ_LEN):
    """Build (tag_ids, correct, label) sequences for KT models."""
    seqs_t, seqs_c, labels = [], [], []
    for uid, grp in df.groupby("user_id"):
        grp = grp.sort_values("timestamp")
        if len(grp) < seq_len + 1:
            continue
        tags = grp["tags"].apply(lambda t: parse_tags(t)[0] if parse_tags(t) else 0).values
        corr = grp["correct"].astype(float).values
        for i in range(0, len(grp) - seq_len - 1, seq_len // 2):
            seqs_t.append(tags[i:i + seq_len])
            seqs_c.append(corr[i:i + seq_len])
            # Label: failure vector for next HORIZON interactions
            lbl = np.zeros(NUM_TAGS, dtype=np.float32)
            for j in range(i + seq_len, min(i + seq_len + HORIZON, len(grp))):
                t = tags[j]
                if 0 < t < NUM_TAGS and corr[j] == 0:
                    lbl[t] = 1.0
            labels.append(lbl)
    return (
        torch.from_numpy(np.array(seqs_t)).long(),
        torch.from_numpy(np.array(seqs_c)).float(),
        torch.from_numpy(np.array(labels)),
    )


def train_kt_model(model_cls, train_df, val_df, seed=42, epochs=15, lr=1e-3, batch_size=128):
    """Generic training loop for DKT/SAKT-style models."""
    set_global_seed(seed)
    model = model_cls().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    t_tags, t_corr, t_labels = build_kt_sequences(train_df)
    v_tags, v_corr, v_labels = build_kt_sequences(val_df)

    train_ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    best_loss = float("inf")
    best_state = None
    patience = 3
    wait = 0

    for epoch in range(epochs):
        model.train()
        for bt, bc, bl in train_loader:
            pred = model(bt.to(DEVICE), bc.to(DEVICE))
            loss = criterion(pred, bl.to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            v_pred = model(v_tags.to(DEVICE), v_corr.to(DEVICE))
            val_loss = criterion(v_pred, v_labels.to(DEVICE)).item()
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    return model


def baseline_kt_model(eval_pairs, model, seq_len=SEQ_LEN):
    """Evaluate a KT model on eval_pairs."""
    preds, scores_list, gt_list, gta_list = [], [], [], []
    model.eval()
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        tags = ctx["tags"].apply(lambda t: parse_tags(t)[0] if parse_tags(t) else 0).values
        corr = ctx["correct"].astype(float).values

        # Pad/truncate to seq_len
        if len(tags) >= seq_len:
            t_seq = tags[-seq_len:]
            c_seq = corr[-seq_len:]
        else:
            t_seq = np.zeros(seq_len, dtype=int)
            c_seq = np.zeros(seq_len, dtype=float)
            t_seq[-len(tags):] = tags
            c_seq[-len(corr):] = corr

        with torch.no_grad():
            t_t = torch.from_numpy(t_seq).long().unsqueeze(0).to(DEVICE)
            c_t = torch.from_numpy(c_seq).float().unsqueeze(0).to(DEVICE)
            scores = model(t_t, c_t).squeeze(0).cpu().numpy()

        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("DKT and SAKT ready.")

In [ ]:
"""Cell 6 — NEW KT baselines: BKT, SAINT, DKT+, Moving Average."""


# ═══════════════════════════════════════════════════════════════
# 1. BKT — Bayesian Knowledge Tracing (per-tag HMM)
# ═══════════════════════════════════════════════════════════════
class BKTModel:
    """
    Bayesian Knowledge Tracing with 4 parameters per tag:
    p_init  — prior probability of mastery
    p_learn — probability of transitioning from unlearned to learned
    p_guess — probability of correct response when unlearned
    p_slip  — probability of incorrect response when learned
    """

    def __init__(self, n_tags=NUM_TAGS):
        self.n_tags = n_tags
        self.p_init = np.full(n_tags, 0.1)
        self.p_learn = np.full(n_tags, 0.1)
        self.p_guess = np.full(n_tags, 0.25)
        self.p_slip = np.full(n_tags, 0.1)

    def fit(self, train_df, n_iter=20):
        """Fit BKT parameters per tag using EM."""
        # Collect per-tag response sequences
        tag_sequences = defaultdict(list)  # tag -> [(user_id, [0/1, ...])]
        for uid, grp in train_df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tag_resps = defaultdict(list)
            for _, row in grp.iterrows():
                for t in parse_tags(row["tags"]):
                    tag_resps[t].append(int(row["correct"]))
            for t, resps in tag_resps.items():
                if len(resps) >= 3 and 0 <= t < self.n_tags:
                    tag_sequences[t].append(resps)

        # EM for each tag
        for tag_id, all_seqs in tag_sequences.items():
            p_i, p_l, p_g, p_s = 0.1, 0.1, 0.25, 0.1
            for _ in range(n_iter):
                sum_gamma0, sum_gamma1 = 0.0, 0.0
                sum_trans_01 = 0.0
                sum_state0 = 0.0
                sum_correct_s0, sum_incorrect_s1 = 0.0, 0.0
                sum_obs_s0, sum_obs_s1 = 0.0, 0.0
                n_init = 0

                for seq in all_seqs:
                    # Forward pass
                    p_know = p_i
                    for obs in seq:
                        # Posterior given observation
                        if obs == 1:
                            p_obs_know = 1 - p_s
                            p_obs_not = p_g
                        else:
                            p_obs_know = p_s
                            p_obs_not = 1 - p_g

                        p_total = p_know * p_obs_know + (1 - p_know) * p_obs_not
                        p_posterior = p_know * p_obs_know / max(p_total, 1e-10)

                        # Accumulate statistics
                        sum_gamma1 += p_posterior
                        sum_gamma0 += (1 - p_posterior)
                        if obs == 1:
                            sum_correct_s0 += (1 - p_posterior)
                            sum_obs_s0 += (1 - p_posterior)
                            sum_obs_s1 += p_posterior
                        else:
                            sum_incorrect_s1 += p_posterior
                            sum_obs_s0 += (1 - p_posterior)
                            sum_obs_s1 += p_posterior

                        # Transition
                        sum_state0 += (1 - p_posterior)
                        p_trans = (1 - p_posterior) * p_l
                        sum_trans_01 += p_trans

                        # Update for next step
                        p_know = p_posterior + (1 - p_posterior) * p_l

                    sum_gamma0 += (1 - p_i)
                    n_init += 1

                # M-step
                if sum_obs_s0 > 0:
                    p_g = np.clip(sum_correct_s0 / sum_obs_s0, 0.01, 0.49)
                if sum_obs_s1 > 0:
                    p_s = np.clip(sum_incorrect_s1 / sum_obs_s1, 0.01, 0.49)
                if sum_state0 > 0:
                    p_l = np.clip(sum_trans_01 / sum_state0, 0.01, 0.99)
                if n_init > 0:
                    p_i = np.clip(sum_gamma1 / (sum_gamma0 + sum_gamma1), 0.01, 0.99)

            self.p_init[tag_id] = p_i
            self.p_learn[tag_id] = p_l
            self.p_guess[tag_id] = p_g
            self.p_slip[tag_id] = p_s

    def predict(self, context_df):
        """Predict P(fail) for each tag given context interactions."""
        # Track mastery per tag
        p_know = self.p_init.copy()
        for _, row in context_df.iterrows():
            for t in parse_tags(row["tags"]):
                if 0 <= t < self.n_tags:
                    obs = int(row["correct"])
                    if obs == 1:
                        p_obs_k = 1 - self.p_slip[t]
                        p_obs_nk = self.p_guess[t]
                    else:
                        p_obs_k = self.p_slip[t]
                        p_obs_nk = 1 - self.p_guess[t]
                    p_total = p_know[t] * p_obs_k + (1 - p_know[t]) * p_obs_nk
                    p_know[t] = p_know[t] * p_obs_k / max(p_total, 1e-10)
                    p_know[t] = p_know[t] + (1 - p_know[t]) * self.p_learn[t]

        # P(fail) = 1 - P(correct)
        p_correct = p_know * (1 - self.p_slip) + (1 - p_know) * self.p_guess
        return (1 - p_correct).astype(np.float32)


def baseline_bkt(eval_pairs, bkt_model):
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        scores = bkt_model.predict(ctx)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════════
# 2. SAINT (Simplified) — Transformer encoder-decoder for KT
# ═══════════════════════════════════════════════════════════════
class SAINTModel(nn.Module):
    """
    Simplified SAINT: Transformer encoder-decoder for Knowledge Tracing.
    Encoder processes exercise (tag) sequences; decoder processes response sequences.
    """

    def __init__(self, n_tags=NUM_TAGS, d_model=64, nhead=4,
                 num_enc_layers=2, num_dec_layers=2, max_len=64, dropout=0.1):
        super().__init__()
        # Encoder: exercise embeddings
        self.tag_emb = nn.Embedding(n_tags + 1, d_model, padding_idx=0)
        self.enc_pos = nn.Embedding(max_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_enc_layers)

        # Decoder: response embeddings
        self.resp_emb = nn.Embedding(3, d_model, padding_idx=0)  # 0=pad, 1=wrong, 2=correct
        self.dec_pos = nn.Embedding(max_len, d_model)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_dec_layers)

        self.fc = nn.Linear(d_model, n_tags)

    def forward(self, tag_ids, correct):
        B, T = tag_ids.size()
        pos = torch.arange(T, device=tag_ids.device).clamp(0, 63).unsqueeze(0).expand(B, -1)

        # Encoder: exercise sequence
        enc_in = self.tag_emb(tag_ids) + self.enc_pos(pos)
        memory = self.encoder(enc_in)

        # Decoder: response sequence (shift by 1 for causal)
        resp_ids = (correct > 0.5).long() + 1  # 1=wrong, 2=correct
        dec_in = self.resp_emb(resp_ids) + self.dec_pos(pos)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=tag_ids.device)
        dec_out = self.decoder(dec_in, memory, tgt_mask=causal_mask)

        logits = self.fc(dec_out[:, -1, :])
        return torch.sigmoid(logits)


# ═══════════════════════════════════════════════════════════════
# 3. DKT+ — DKT with self-attention (DKT + Attention)
# ═══════════════════════════════════════════════════════════════
class DKTPlusModel(nn.Module):
    """DKT with self-attention layer on top of LSTM."""

    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, hidden=64, nhead=4):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags + 1, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim + 1, hidden, batch_first=True)
        self.attn = nn.MultiheadAttention(hidden, nhead, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(hidden)
        self.fc = nn.Linear(hidden, n_tags)

    def forward(self, tag_ids, correct):
        emb = self.tag_emb(tag_ids)
        x = torch.cat([emb, correct.unsqueeze(-1)], dim=-1)
        lstm_out, _ = self.lstm(x)
        # Self-attention over LSTM outputs
        causal = nn.Transformer.generate_square_subsequent_mask(
            lstm_out.size(1), device=lstm_out.device,
        )
        attn_out, _ = self.attn(lstm_out, lstm_out, lstm_out, attn_mask=causal)
        out = self.norm(lstm_out + attn_out)
        return torch.sigmoid(self.fc(out[:, -1, :]))


# ═══════════════════════════════════════════════════════════════
# 4. Moving Average — naive per-tag accuracy baseline
# ═══════════════════════════════════════════════════════════════
def baseline_moving_average(eval_pairs, window=50, threshold=0.6):
    """
    Predict gap if accuracy on last `window` interactions per tag < threshold.
    Score = 1 - moving_accuracy (higher = bigger gap).
    """
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        tag_correct = defaultdict(list)
        for _, row in ctx.iterrows():
            for t in parse_tags(row["tags"]):
                tag_correct[t].append(int(row["correct"]))

        scores = np.full(NUM_TAGS, 0.5, dtype=np.float32)  # Default: uncertain
        for t, resps in tag_correct.items():
            if 0 <= t < NUM_TAGS:
                recent = resps[-window:]
                acc = np.mean(recent)
                scores[t] = 1.0 - acc  # Higher score = bigger gap

        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("NEW KT baselines ready: BKT, SAINT, DKT+, Moving Average")

## Recommendation Baselines

BPR-MF (implicit), NeuMF (PyTorch), KG-based RS (without multi-agent).

In [ ]:
"""Cell 7 — NEW Recommendation baselines: BPR-MF, NeuMF, KG-RS."""


# ═══════════════════════════════════════════════════════════════
# 5. BPR-MF — Bayesian Personalized Ranking with Matrix Factorization
# ═══════════════════════════════════════════════════════════════
def train_bpr_mf(train_df, n_factors=64, iterations=50, seed=42):
    """Train BPR-MF using the implicit library (ALS fallback if BPR unavailable)."""
    from scipy.sparse import csr_matrix
    import implicit

    user_ids = train_df["user_id"].unique()
    uid_map = {u: i for i, u in enumerate(user_ids)}
    n_users = len(user_ids)

    # Build user-tag interaction matrix (implicit feedback: failure counts)
    rows, cols, data = [], [], []
    for uid, grp in train_df.groupby("user_id"):
        u_idx = uid_map[uid]
        for _, row in grp.iterrows():
            for t in parse_tags(row["tags"]):
                if not row["correct"] and 0 <= t < NUM_TAGS:
                    rows.append(u_idx)
                    cols.append(t)
                    data.append(1.0)

    mat = csr_matrix((data, (rows, cols)), shape=(n_users, NUM_TAGS), dtype=np.float32)

    # Try BPR, fall back to ALS
    try:
        model = implicit.bpr.BayesianPersonalizedRanking(
            factors=n_factors, iterations=iterations, random_state=seed,
        )
    except AttributeError:
        model = implicit.als.AlternatingLeastSquares(
            factors=n_factors, iterations=iterations, random_state=seed,
        )

    model.fit(mat)
    return model, uid_map, mat


def baseline_bpr_mf(eval_pairs, model, uid_map, mat):
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        if uid in uid_map:
            u_idx = uid_map[uid]
            ids, scores_raw = model.recommend(u_idx, mat[u_idx], N=NUM_TAGS, filter_already_liked_items=False)
            scores = np.zeros(NUM_TAGS, dtype=np.float32)
            for tag_id, score in zip(ids, scores_raw):
                if 0 <= tag_id < NUM_TAGS:
                    scores[tag_id] = score
        else:
            scores = np.zeros(NUM_TAGS, dtype=np.float32)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════════
# 6. NeuMF / NCF — Neural Collaborative Filtering
# ═══════════════════════════════════════════════════════════════
class NeuMF(nn.Module):
    """Neural Matrix Factorization: GMF + MLP fusion."""

    def __init__(self, n_users, n_items=NUM_TAGS, emb_dim=32, mlp_dims=(64, 32)):
        super().__init__()
        # GMF path
        self.gmf_user = nn.Embedding(n_users, emb_dim)
        self.gmf_item = nn.Embedding(n_items, emb_dim)

        # MLP path
        self.mlp_user = nn.Embedding(n_users, emb_dim)
        self.mlp_item = nn.Embedding(n_items, emb_dim)

        layers = []
        in_dim = emb_dim * 2
        for out_dim in mlp_dims:
            layers.extend([nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(0.2)])
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)

        # Fusion
        self.output = nn.Linear(emb_dim + mlp_dims[-1], 1)

    def forward(self, user_ids, item_ids):
        # GMF
        gmf = self.gmf_user(user_ids) * self.gmf_item(item_ids)
        # MLP
        mlp_in = torch.cat([self.mlp_user(user_ids), self.mlp_item(item_ids)], dim=-1)
        mlp_out = self.mlp(mlp_in)
        # Fuse
        fused = torch.cat([gmf, mlp_out], dim=-1)
        return torch.sigmoid(self.output(fused)).squeeze(-1)


def train_neumf(train_df, seed=42, epochs=15, batch_size=512, neg_ratio=4):
    """Train NeuMF on implicit feedback (failures)."""
    set_global_seed(seed)
    user_ids = train_df["user_id"].unique()
    uid_map = {u: i for i, u in enumerate(user_ids)}
    n_users = len(user_ids)

    # Positive pairs: (user, tag) where user failed on tag
    pos_pairs = set()
    for _, row in train_df.iterrows():
        if not row["correct"]:
            u = uid_map.get(row["user_id"])
            if u is not None:
                for t in parse_tags(row["tags"]):
                    if 0 <= t < NUM_TAGS:
                        pos_pairs.add((u, t))

    pos_list = list(pos_pairs)
    if len(pos_list) == 0:
        return None, uid_map

    # Build training data with negative sampling
    rng = np.random.RandomState(seed)
    users, items, labels = [], [], []
    for u, t in pos_list:
        users.append(u)
        items.append(t)
        labels.append(1.0)
        # Negative samples
        for _ in range(neg_ratio):
            neg_t = rng.randint(0, NUM_TAGS)
            while (u, neg_t) in pos_pairs:
                neg_t = rng.randint(0, NUM_TAGS)
            users.append(u)
            items.append(neg_t)
            labels.append(0.0)

    ds = torch.utils.data.TensorDataset(
        torch.tensor(users, dtype=torch.long),
        torch.tensor(items, dtype=torch.long),
        torch.tensor(labels, dtype=torch.float32),
    )
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)

    model = NeuMF(n_users).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        for bu, bi, bl in loader:
            pred = model(bu.to(DEVICE), bi.to(DEVICE))
            loss = criterion(pred, bl.to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    return model, uid_map


def baseline_neumf(eval_pairs, model, uid_map):
    preds, scores_list, gt_list, gta_list = [], [], [], []
    all_items = torch.arange(NUM_TAGS, device=DEVICE)

    for uid, ctx, gt, gt_all, _ in eval_pairs:
        if model is None or uid not in uid_map:
            scores = np.zeros(NUM_TAGS, dtype=np.float32)
        else:
            u_idx = uid_map[uid]
            with torch.no_grad():
                u_tensor = torch.full((NUM_TAGS,), u_idx, dtype=torch.long, device=DEVICE)
                scores = model(u_tensor, all_items).cpu().numpy().astype(np.float32)

        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════════
# 7. KG-based RS — GraphSAGE embeddings + cosine similarity
# ═══════════════════════════════════════════════════════════════
def baseline_kg_rs(eval_pairs, train_df=None, seed=42):
    """
    KG-based recommendation WITHOUT multi-agent architecture.
    Uses GraphSAGE tag embeddings + cosine similarity for ranking.
    No confidence features, no TS, no LambdaMART.
    """
    from agents.kg_agent import KnowledgeGraphAgent

    set_global_seed(seed)
    kg = KnowledgeGraphAgent()
    kg.build_graph(questions_df, lectures_df)
    kg.build_prerequisites(train_df)
    if hasattr(kg, "train_embeddings"):
        try:
            kg.train_embeddings()
        except Exception:
            pass

    # Get tag embeddings
    tag_embs = getattr(kg, "tag_embeddings", None)
    if tag_embs is None:
        tag_embs = np.random.randn(NUM_TAGS, 64).astype(np.float32)

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        # Build user profile from context: average embeddings of failed tags
        failed_tags = []
        for _, row in ctx.iterrows():
            if not row["correct"]:
                for t in parse_tags(row["tags"]):
                    if 0 <= t < NUM_TAGS:
                        failed_tags.append(t)

        if failed_tags:
            user_emb = np.mean(tag_embs[failed_tags], axis=0)
            # Cosine similarity with all tags
            norms = np.linalg.norm(tag_embs, axis=1, keepdims=True) + 1e-8
            user_norm = np.linalg.norm(user_emb) + 1e-8
            scores = (tag_embs @ user_emb) / (norms.squeeze() * user_norm)
            scores = scores.astype(np.float32)
        else:
            scores = np.zeros(NUM_TAGS, dtype=np.float32)

        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("NEW Recommendation baselines ready: BPR-MF, NeuMF, KG-RS")

## Bandit Baselines

e-greedy, UCB1, Fixed — alternatives to Thompson Sampling for strategy selection.

In [ ]:
"""Cell 8 — Bandit baselines: e-greedy, UCB1, Fixed + evaluation framework."""

STRATEGIES = ["knowledge_based", "content_based", "collaborative"]


class BanditSimulator:
    """
    Simulates bandit strategy selection over a sequence of interactions.
    Each strategy produces a "reward" (1 if recommended tag matches actual gap, 0 otherwise).
    """

    def __init__(self, train_df, eval_pairs, seed=42):
        self.rng = np.random.RandomState(seed)
        self.eval_pairs = eval_pairs

        # Pre-compute per-strategy scores for each user
        # knowledge_based: user failure rate per tag
        # content_based: tag difficulty
        # collaborative: SVD-based scores
        self.strategy_scores = self._precompute_strategies(train_df, seed)

    def _precompute_strategies(self, train_df, seed):
        """Pre-compute scores for each strategy."""
        scores = {}

        # Knowledge-based: per-user failure rate
        user_tag_fail = defaultdict(lambda: np.zeros(NUM_TAGS, dtype=np.float32))
        user_tag_total = defaultdict(lambda: np.zeros(NUM_TAGS, dtype=np.float32))
        for _, row in train_df.iterrows():
            uid = str(row["user_id"])
            for t in parse_tags(row["tags"]):
                user_tag_total[uid][t] += 1
                if not row["correct"]:
                    user_tag_fail[uid][t] += 1
        scores["knowledge_based"] = {}
        for uid in user_tag_fail:
            mask = user_tag_total[uid] > 0
            s = np.zeros(NUM_TAGS, dtype=np.float32)
            s[mask] = user_tag_fail[uid][mask] / user_tag_total[uid][mask]
            scores["knowledge_based"][uid] = s

        # Content-based: global tag difficulty
        tag_fail = np.zeros(NUM_TAGS, dtype=np.float32)
        tag_total = np.zeros(NUM_TAGS, dtype=np.float32)
        for _, row in train_df.iterrows():
            for t in parse_tags(row["tags"]):
                tag_total[t] += 1
                if not row["correct"]:
                    tag_fail[t] += 1
        tag_diff = np.divide(tag_fail, tag_total, out=np.full(NUM_TAGS, 0.5),
                             where=tag_total > 0)
        scores["content_based"] = tag_diff

        # Collaborative: SVD
        from sklearn.decomposition import TruncatedSVD
        from scipy.sparse import lil_matrix
        user_ids = train_df["user_id"].unique()
        uid_idx = {str(u): i for i, u in enumerate(user_ids)}
        mat = lil_matrix((len(user_ids), NUM_TAGS), dtype=np.float32)
        for uid, grp in train_df.groupby("user_id"):
            for _, row in grp.iterrows():
                for t in parse_tags(row["tags"]):
                    if not row["correct"]:
                        mat[uid_idx[str(uid)], t] += 1.0
        n_comp = min(32, min(mat.shape) - 1)
        svd = TruncatedSVD(n_components=n_comp, random_state=seed)
        user_f = svd.fit_transform(mat.tocsr())
        item_f = svd.components_.T
        scores["collaborative"] = {"uid_idx": uid_idx, "user_f": user_f, "item_f": item_f}

        return scores

    def get_reward(self, uid, strategy, gt):
        """Get reward for a strategy selection: fraction of gt tags in top-10."""
        if strategy == "knowledge_based":
            s = self.strategy_scores["knowledge_based"].get(uid, np.zeros(NUM_TAGS))
        elif strategy == "content_based":
            s = self.strategy_scores["content_based"]
        elif strategy == "collaborative":
            cf = self.strategy_scores["collaborative"]
            if uid in cf["uid_idx"]:
                idx = cf["uid_idx"][uid]
                s = (cf["item_f"] @ cf["user_f"][idx]).astype(np.float32)
            else:
                s = np.zeros(NUM_TAGS, dtype=np.float32)
        else:
            s = np.zeros(NUM_TAGS, dtype=np.float32)

        top_k = np.argsort(s)[::-1][:TOP_K]
        hits = len(set(top_k) & gt)
        return hits / max(len(gt), 1)

    def run_bandit(self, policy_fn, n_rounds=None):
        """
        Run bandit simulation and return cumulative reward + regret.

        policy_fn(step, history) -> strategy_name
        """
        if n_rounds is None:
            n_rounds = len(self.eval_pairs)
        else:
            n_rounds = min(n_rounds, len(self.eval_pairs))

        cum_reward = 0.0
        cum_regret = 0.0
        rewards_history = []
        regrets_history = []
        strategy_history = []

        for step in range(n_rounds):
            uid, ctx, gt, gt_all, _ = self.eval_pairs[step % len(self.eval_pairs)]

            # Select strategy
            strategy = policy_fn(step, rewards_history)
            reward = self.get_reward(uid, strategy, gt)

            # Oracle reward (best strategy)
            best_reward = max(self.get_reward(uid, s, gt) for s in STRATEGIES)
            regret = best_reward - reward

            cum_reward += reward
            cum_regret += regret
            rewards_history.append((strategy, reward))
            regrets_history.append(cum_regret)
            strategy_history.append(strategy)

        return {
            "cumulative_reward": round(cum_reward, 4),
            "cumulative_regret": round(cum_regret, 4),
            "avg_reward": round(cum_reward / n_rounds, 4),
            "avg_regret": round(cum_regret / n_rounds, 4),
            "strategy_counts": {s: strategy_history.count(s) for s in STRATEGIES},
            "regret_curve": regrets_history,
            "convergence_step": self._find_convergence(rewards_history),
        }

    @staticmethod
    def _find_convergence(history, window=50, threshold=0.01):
        """Find step where rolling average reward stabilizes."""
        if len(history) < window * 2:
            return len(history)
        rewards = [r for _, r in history]
        rolling = pd.Series(rewards).rolling(window).mean().values
        for i in range(window, len(rolling) - window):
            if abs(rolling[i] - rolling[i + window]) < threshold:
                return i
        return len(history)


# --- Policy implementations ---

def policy_thompson_sampling(step, history, rng=None, priors=None):
    if priors is None:
        priors = {s: {"alpha": 1.0, "beta": 1.0} for s in STRATEGIES}
    if rng is None:
        rng = np.random.RandomState(42)
    # Update from history
    params = {s: dict(p) for s, p in priors.items()}
    for strat, reward in history:
        params[strat]["alpha"] += reward
        params[strat]["beta"] += (1.0 - reward)
    # Sample
    best_s, best_v = STRATEGIES[0], -1
    for s in STRATEGIES:
        v = rng.beta(params[s]["alpha"], params[s]["beta"])
        if v > best_v:
            best_v = v
            best_s = s
    return best_s


def policy_epsilon_greedy(step, history, epsilon=0.1, rng=None):
    if rng is None:
        rng = np.random.RandomState(42)
    if rng.random() < epsilon or len(history) == 0:
        return STRATEGIES[rng.randint(len(STRATEGIES))]
    # Exploit: pick strategy with highest average reward
    strat_rewards = defaultdict(list)
    for s, r in history:
        strat_rewards[s].append(r)
    return max(STRATEGIES, key=lambda s: np.mean(strat_rewards[s]) if strat_rewards[s] else 0)


def policy_ucb1(step, history):
    strat_rewards = defaultdict(list)
    for s, r in history:
        strat_rewards[s].append(r)
    total = len(history) + 1
    best_s, best_v = STRATEGIES[0], -1
    for s in STRATEGIES:
        if not strat_rewards[s]:
            return s  # Explore unvisited
        avg = np.mean(strat_rewards[s])
        exploration = np.sqrt(2 * np.log(total) / len(strat_rewards[s]))
        v = avg + exploration
        if v > best_v:
            best_v = v
            best_s = s
    return best_s


def policy_fixed_knowledge(step, history):
    return "knowledge_based"


print("Bandit baselines ready: Thompson Sampling, e-greedy, UCB1, Fixed")

In [ ]:
"""Cell 9 — MARS (ours) baseline wrapper."""

from agents.prediction_agent import PredictionAgent
from agents.confidence_agent import ConfidenceAgent
from agents.diagnostic_agent import DiagnosticAgent


def baseline_mars(eval_pairs, train_df, val_df, seed=42):
    """Run full MARS pipeline as a baseline for fair comparison."""
    set_global_seed(seed)

    # Train agents
    diag = DiagnosticAgent(seed=seed)
    irt_params = diag.calibrate_from_interactions(train_df, min_answers_per_q=20)

    conf = ConfidenceAgent()
    conf.train(train_df, irt_params=irt_params, use_class_weight=True)

    pred = PredictionAgent()
    pred.train(train_df, val_df=val_df)

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        # Classify confidence
        conf_result = conf.classify_batch(uid, ctx)

        # Enrich context with confidence
        ctx_copy = ctx.copy()
        if isinstance(conf_result, dict) and "predictions" in conf_result:
            classes = conf_result["predictions"]
            if len(classes) == len(ctx_copy):
                ctx_copy["confidence_class"] = [c.get("class", 0) if isinstance(c, dict) else 0
                                                 for c in classes]
            else:
                ctx_copy["confidence_class"] = 0
        else:
            ctx_copy["confidence_class"] = 0

        # Predict gaps
        result = pred.predict_gaps(uid, recent=ctx_copy, threshold=0.0)
        scores = np.array(result.get("gap_probabilities", np.zeros(NUM_TAGS)),
                          dtype=np.float32)
        if len(scores) != NUM_TAGS:
            scores = np.zeros(NUM_TAGS, dtype=np.float32)

        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)

    return preds, scores_list, gt_list, gta_list


print("MARS baseline wrapper ready.")

## Run All Baselines (5 seeds)

Tables 1 & 2 share the same eval loop. Table 3 (bandits) runs separately.

In [ ]:
"""Cell 10 — Run all prediction + recommendation baselines (5 seeds)."""

all_results = defaultdict(list)  # method -> [metrics_dict per seed]
cache_path = RESULTS_DIR / "baselines_raw.json"

# Resume from cache
if cache_path.exists():
    with open(cache_path) as f:
        cached = json.load(f)
    all_results = defaultdict(list, {k: v for k, v in cached.items()})
    done_seeds = {
        method: len(results) for method, results in all_results.items()
    }
    print(f"Loaded cache: {dict(done_seeds)}")
else:
    done_seeds = {}

for seed_idx, seed in enumerate(SEEDS):
    logger.info("=" * 60)
    logger.info("SEED %d/%d: %d", seed_idx + 1, len(SEEDS), seed)
    logger.info("=" * 60)

    set_global_seed(seed)
    rng = np.random.RandomState(seed)

    # --- Non-trainable baselines ---
    for name, fn, kwargs in [
        ("Random", baseline_random, {"seed": seed}),
        ("Popular", baseline_popular, {"train_df": train_df}),
        ("Content-only", baseline_content_only, {"train_df": train_df, "seed": seed}),
        ("CF-only (SVD)", baseline_cf_only, {"train_df": train_df, "seed": seed}),
        ("Moving Average", baseline_moving_average, {}),
    ]:
        if len(all_results.get(name, [])) > seed_idx:
            continue
        logger.info("  Running %s...", name)
        t0 = time.time()
        p, s, g, ga = fn(eval_pairs, **kwargs)
        m = compute_all_metrics(p, s, g, ga)
        m["time_s"] = round(time.time() - t0, 1)
        all_results[name].append(m)

    # --- IRT-only ---
    if len(all_results.get("IRT-only", [])) <= seed_idx:
        logger.info("  Running IRT-only...")
        t0 = time.time()
        p, s, g, ga = baseline_irt_only(eval_pairs, train_df=train_df, seed=seed)
        m = compute_all_metrics(p, s, g, ga)
        m["time_s"] = round(time.time() - t0, 1)
        all_results["IRT-only"].append(m)

    # --- KT models (need training) ---
    for name, model_cls in [
        ("DKT", DKTModel),
        ("SAKT", SAKTModel),
        ("SAINT", SAINTModel),
        ("DKT+", DKTPlusModel),
    ]:
        if len(all_results.get(name, [])) > seed_idx:
            continue
        logger.info("  Training + running %s...", name)
        t0 = time.time()
        model = train_kt_model(model_cls, train_df, val_df, seed=seed)
        p, s, g, ga = baseline_kt_model(eval_pairs, model)
        m = compute_all_metrics(p, s, g, ga)
        m["time_s"] = round(time.time() - t0, 1)
        all_results[name].append(m)
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # --- BKT ---
    if len(all_results.get("BKT", [])) <= seed_idx:
        logger.info("  Training + running BKT...")
        t0 = time.time()
        bkt = BKTModel()
        bkt.fit(train_df)
        p, s, g, ga = baseline_bkt(eval_pairs, bkt)
        m = compute_all_metrics(p, s, g, ga)
        m["time_s"] = round(time.time() - t0, 1)
        all_results["BKT"].append(m)

    # --- BPR-MF ---
    if len(all_results.get("BPR-MF", [])) <= seed_idx:
        logger.info("  Training + running BPR-MF...")
        t0 = time.time()
        try:
            bpr_model, bpr_uid_map, bpr_mat = train_bpr_mf(train_df, seed=seed)
            p, s, g, ga = baseline_bpr_mf(eval_pairs, bpr_model, bpr_uid_map, bpr_mat)
            m = compute_all_metrics(p, s, g, ga)
        except Exception as e:
            logger.warning("  BPR-MF failed: %s", e)
            m = {k: 0.0 for k in ["auc_roc", "auc_pr", "f1_macro", "ndcg@5", "ndcg@10",
                                    "precision@10", "recall@10", "map", "mrr", "coverage"]}
        m["time_s"] = round(time.time() - t0, 1)
        all_results["BPR-MF"].append(m)

    # --- NeuMF ---
    if len(all_results.get("NeuMF", [])) <= seed_idx:
        logger.info("  Training + running NeuMF...")
        t0 = time.time()
        neumf_model, neumf_uid_map = train_neumf(train_df, seed=seed)
        p, s, g, ga = baseline_neumf(eval_pairs, neumf_model, neumf_uid_map)
        m = compute_all_metrics(p, s, g, ga)
        m["time_s"] = round(time.time() - t0, 1)
        all_results["NeuMF"].append(m)
        del neumf_model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # --- KG-RS ---
    if len(all_results.get("KG-RS", [])) <= seed_idx:
        logger.info("  Training + running KG-RS...")
        t0 = time.time()
        p, s, g, ga = baseline_kg_rs(eval_pairs, train_df=train_df, seed=seed)
        m = compute_all_metrics(p, s, g, ga)
        m["time_s"] = round(time.time() - t0, 1)
        all_results["KG-RS"].append(m)

    # --- MARS (ours) ---
    if len(all_results.get("MARS (ours)", [])) <= seed_idx:
        logger.info("  Training + running MARS...")
        t0 = time.time()
        p, s, g, ga = baseline_mars(eval_pairs, train_df, val_df, seed=seed)
        m = compute_all_metrics(p, s, g, ga)
        m["time_s"] = round(time.time() - t0, 1)
        all_results["MARS (ours)"].append(m)

    # Save checkpoint
    with open(cache_path, "w") as f:
        json.dump(dict(all_results), f, indent=2)
    logger.info("  Checkpoint saved.")

print(f"\nAll baselines complete. Methods: {list(all_results.keys())}")
print(f"Seeds per method: {[(m, len(v)) for m, v in all_results.items()]}")

In [ ]:
"""Cell 11 — Table 1: Knowledge Gap Prediction."""


def aggregate_method(results_list, metric):
    """Extract mean ± std for a metric across seeds."""
    vals = [r.get(metric, 0.0) for r in results_list]
    if not vals:
        return 0.0, 0.0, vals
    return float(np.mean(vals)), float(np.std(vals, ddof=1) if len(vals) > 1 else 0.0), vals


def significance_marker(mars_vals, baseline_vals, bonferroni_n=1):
    """Wilcoxon test with Bonferroni correction, return marker string."""
    if len(mars_vals) < 5 or len(baseline_vals) < 5:
        return ""
    if len(mars_vals) != len(baseline_vals):
        return ""
    try:
        _, p = wilcoxon(mars_vals, baseline_vals)
        p_adj = min(p * bonferroni_n, 1.0)
        if p_adj < 0.001:
            return "***"
        elif p_adj < 0.01:
            return "**"
        elif p_adj < 0.05:
            return "*"
    except ValueError:
        pass
    return ""


# --- Table 1: KT methods ---
kt_methods = ["Moving Average", "BKT", "IRT-only", "DKT", "SAKT", "DKT+", "SAINT", "MARS (ours)"]
kt_metrics = ["auc_roc", "auc_pr", "f1_macro"]

mars_results = all_results.get("MARS (ours)", [])
n_kt = len(kt_methods) - 1  # Bonferroni correction factor

table1_rows = []
for method in kt_methods:
    results = all_results.get(method, [])
    row = {"Method": method}
    for metric in kt_metrics:
        mean, std, vals = aggregate_method(results, metric)
        sig = significance_marker(
            [r.get(metric, 0) for r in mars_results],
            vals, bonferroni_n=n_kt,
        ) if method != "MARS (ours)" else ""
        # Bold MARS
        if method == "MARS (ours)":
            row[metric.upper()] = f"**{mean:.4f}** ± {std:.4f}"
        else:
            row[metric.upper()] = f"{mean:.4f} ± {std:.4f} {sig}".strip()
    table1_rows.append(row)

table1 = pd.DataFrame(table1_rows)
print("=" * 90)
print("TABLE 1: Knowledge Gap Prediction (mean ± std, 5 seeds, Wilcoxon + Bonferroni)")
print("=" * 90)
display(table1)
table1.to_csv(RESULTS_DIR / "table1_gap_prediction.csv", index=False)
table1.to_latex(RESULTS_DIR / "table1_gap_prediction.tex", index=False, escape=False)

In [ ]:
"""Cell 12 — Table 2: Content Recommendation."""

rec_methods = [
    "Random", "Popular", "Content-only", "CF-only (SVD)", "BPR-MF", "NeuMF",
    "KG-RS", "IRT-only", "DKT", "SAKT", "SAINT", "DKT+", "MARS (ours)",
]
rec_metrics = ["ndcg@5", "ndcg@10", "precision@10", "recall@10", "map", "mrr", "coverage"]

n_rec = len(rec_methods) - 1

table2_rows = []
for method in rec_methods:
    results = all_results.get(method, [])
    row = {"Method": method}
    for metric in rec_metrics:
        mean, std, vals = aggregate_method(results, metric)
        sig = significance_marker(
            [r.get(metric, 0) for r in mars_results],
            vals, bonferroni_n=n_rec,
        ) if method != "MARS (ours)" else ""
        col_name = metric.upper().replace("@", "@")
        if method == "MARS (ours)":
            row[col_name] = f"**{mean:.4f}** ± {std:.4f}"
        else:
            row[col_name] = f"{mean:.4f} ± {std:.4f} {sig}".strip()
    table2_rows.append(row)

table2 = pd.DataFrame(table2_rows)
print("=" * 120)
print("TABLE 2: Content Recommendation (mean ± std, 5 seeds, Wilcoxon + Bonferroni)")
print("=" * 120)
display(table2)
table2.to_csv(RESULTS_DIR / "table2_recommendation.csv", index=False)
table2.to_latex(RESULTS_DIR / "table2_recommendation.tex", index=False, escape=False)

In [ ]:
"""Cell 13 — Table 3: Strategy Selection (Bandit comparison)."""

bandit_results = defaultdict(list)

for seed_idx, seed in enumerate(SEEDS):
    set_global_seed(seed)
    rng = np.random.RandomState(seed)
    sim = BanditSimulator(train_df, eval_pairs, seed=seed)

    policies = {
        "Thompson Sampling": lambda s, h: policy_thompson_sampling(s, h, rng=rng),
        "e-greedy (e=0.1)": lambda s, h: policy_epsilon_greedy(s, h, epsilon=0.1, rng=rng),
        "UCB1": policy_ucb1,
        "Fixed (knowledge)": policy_fixed_knowledge,
    }

    for name, policy_fn in policies.items():
        result = sim.run_bandit(policy_fn)
        bandit_results[name].append(result)
        logger.info("  [seed=%d] %s: reward=%.4f, regret=%.4f, convergence=%d",
                     seed, name, result["avg_reward"], result["avg_regret"],
                     result["convergence_step"])

# Build Table 3
table3_rows = []
for name in ["Thompson Sampling", "e-greedy (e=0.1)", "UCB1", "Fixed (knowledge)"]:
    results = bandit_results[name]
    cum_rewards = [r["cumulative_reward"] for r in results]
    cum_regrets = [r["cumulative_regret"] for r in results]
    convergences = [r["convergence_step"] for r in results]

    row = {
        "Strategy": name,
        "Cumulative Reward": f"{np.mean(cum_rewards):.1f} ± {np.std(cum_rewards, ddof=1):.1f}",
        "Cumulative Regret": f"{np.mean(cum_regrets):.1f} ± {np.std(cum_regrets, ddof=1):.1f}",
        "Avg Reward": f"{np.mean([r['avg_reward'] for r in results]):.4f}",
        "Convergence (steps)": f"{int(np.mean(convergences))}",
    }
    table3_rows.append(row)

table3 = pd.DataFrame(table3_rows)
print("=" * 100)
print("TABLE 3: Strategy Selection — Bandit Comparison (mean ± std, 5 seeds)")
print("=" * 100)
display(table3)
table3.to_csv(RESULTS_DIR / "table3_bandits.csv", index=False)
table3.to_latex(RESULTS_DIR / "table3_bandits.tex", index=False, escape=False)

## Visualizations

In [ ]:
"""Cell 14 — Visualization: NDCG@10 bar chart + Regret curves."""

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Panel A: NDCG@10 comparison across all methods ---
ax = axes[0]
methods_sorted = sorted(
    all_results.keys(),
    key=lambda m: np.mean([r.get("ndcg@10", 0) for r in all_results[m]]),
)
means = [np.mean([r.get("ndcg@10", 0) for r in all_results[m]]) for m in methods_sorted]
stds = [np.std([r.get("ndcg@10", 0) for r in all_results[m]]) for m in methods_sorted]
colors = ["#d62728" if m == "MARS (ours)" else "#1f77b4" for m in methods_sorted]

y = np.arange(len(methods_sorted))
ax.barh(y, means, xerr=stds, color=colors, edgecolor="black", linewidth=0.5,
        capsize=3, height=0.6)
ax.set_yticks(y)
ax.set_yticklabels(methods_sorted, fontsize=9)
ax.set_xlabel("NDCG@10", fontsize=11)
ax.set_title("(a) NDCG@10 — All Methods", fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(m + 0.002, i, f"{m:.3f}", va="center", fontsize=8)

# --- Panel B: Bandit regret curves ---
ax2 = axes[1]
policy_colors = {
    "Thompson Sampling": "#1f77b4",
    "e-greedy (e=0.1)": "#ff7f0e",
    "UCB1": "#2ca02c",
    "Fixed (knowledge)": "#d62728",
}

for name in ["Thompson Sampling", "e-greedy (e=0.1)", "UCB1", "Fixed (knowledge)"]:
    results = bandit_results[name]
    # Average regret curves across seeds
    min_len = min(len(r["regret_curve"]) for r in results)
    curves = np.array([r["regret_curve"][:min_len] for r in results])
    mean_curve = curves.mean(axis=0)
    std_curve = curves.std(axis=0)
    steps = np.arange(min_len)
    ax2.plot(steps, mean_curve, label=name, color=policy_colors[name], linewidth=1.5)
    ax2.fill_between(steps, mean_curve - std_curve, mean_curve + std_curve,
                     color=policy_colors[name], alpha=0.15)

ax2.set_xlabel("Steps", fontsize=11)
ax2.set_ylabel("Cumulative Regret", fontsize=11)
ax2.set_title("(b) Strategy Selection — Regret Curves", fontsize=12, fontweight="bold")
ax2.legend(loc="upper left", fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "baselines_comparison.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "baselines_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved: baselines_comparison.png / .pdf")

In [ ]:
"""Cell 15 — AUC-ROC heatmap + summary statistics."""

# --- Heatmap: all methods × key metrics ---
heatmap_methods = list(all_results.keys())
heatmap_metrics = ["auc_roc", "ndcg@10", "precision@10", "mrr", "coverage"]

data = []
for method in heatmap_methods:
    row = {}
    for metric in heatmap_metrics:
        vals = [r.get(metric, 0) for r in all_results[method]]
        row[metric.upper()] = np.mean(vals) if vals else 0.0
    data.append(row)

heatmap_df = pd.DataFrame(data, index=heatmap_methods)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    heatmap_df, annot=True, fmt=".3f", cmap="YlOrRd",
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "Metric Value"},
)
ax.set_title("Baseline Comparison — Key Metrics Heatmap", fontsize=13, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baselines_heatmap.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "baselines_heatmap.pdf", bbox_inches="tight")
plt.show()

# --- Summary statistics ---
print("\n" + "=" * 80)
print("SUMMARY: MARS improvement over best non-MARS baseline")
print("=" * 80)
for metric in ["auc_roc", "ndcg@10", "precision@10", "mrr"]:
    mars_mean = np.mean([r.get(metric, 0) for r in all_results.get("MARS (ours)", [])])
    best_other = 0
    best_name = ""
    for method, results in all_results.items():
        if method == "MARS (ours)":
            continue
        m = np.mean([r.get(metric, 0) for r in results])
        if m > best_other:
            best_other = m
            best_name = method
    if best_other > 0:
        improvement = (mars_mean - best_other) / best_other * 100
        print(f"  {metric.upper():>15}: MARS={mars_mean:.4f}, Best={best_name} ({best_other:.4f}), "
              f"Improvement: {improvement:+.1f}%")

In [ ]:
"""Cell 16 — Save all results."""

final_meta = {
    "experiment": "MARS Extended Baseline Comparison",
    "seeds": SEEDS,
    "n_baselines": len(all_results),
    "methods": list(all_results.keys()),
    "tables": {
        "table1": "Knowledge Gap Prediction (KT methods)",
        "table2": "Content Recommendation (all methods)",
        "table3": "Strategy Selection (bandit comparison)",
    },
    "output_files": [
        str(RESULTS_DIR / "baselines_raw.json"),
        str(RESULTS_DIR / "table1_gap_prediction.csv"),
        str(RESULTS_DIR / "table1_gap_prediction.tex"),
        str(RESULTS_DIR / "table2_recommendation.csv"),
        str(RESULTS_DIR / "table2_recommendation.tex"),
        str(RESULTS_DIR / "table3_bandits.csv"),
        str(RESULTS_DIR / "table3_bandits.tex"),
        str(RESULTS_DIR / "baselines_comparison.png"),
        str(RESULTS_DIR / "baselines_heatmap.png"),
    ],
}

with open(RESULTS_DIR / "baselines_meta.json", "w") as f:
    json.dump(final_meta, f, indent=2)

print("Baseline comparison complete.")
print(f"Total methods: {len(all_results)}")
print(f"Results: {RESULTS_DIR}")